# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # do not treat as dict
print(f"Dataset Title: {getattr(metadata, 'name', 'Unnamed')}")
print(f"Description: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and fields with their @id
def get_record_set_overview(ds):
    print("Available record sets and their fields:")
    for rs in ds.metadata.record_sets:
        print(f"- Record Set: {rs['@id']}  (name: {rs.get('name', None)})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            print(f"    - Field: {f['@id']}  (name: {f.get('name', None)}, dataType: {f.get('dataType', None)})")

get_record_set_overview(dataset)

# Preview a few records from the first record set
if dataset.metadata.record_sets:
    first_record_set_id = dataset.metadata.record_sets[0]['@id']
    print(f"\nSample records from Record Set: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract all record sets
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns for the first record set
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id:
    print(f"Columns in record set {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations like removing outliers, transforming data distributions, or grouping data by key attributes prepare it for further analysis.

In [ ]:
# Choose a numeric field and a group field from the primary record set
numeric_candidates = []
group_candidates = []

if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Guess fields based on their names
    for col in df.columns:
        # Typical numeric field candidates in proteomics
        if any(s in col.lower() for s in ['intensity', 'abundance', 'count', 'mw', 'weight', 'coverage']):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_candidates.append(col)
        if any(s in col.lower() for s in ['group', 'condition', 'type', 'class']):
            group_candidates.append(col)
    # Default to first numeric column if none found
    if not numeric_candidates:
        for col in df.select_dtypes('number').columns:
            numeric_candidates.append(col)
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {filtered_df.shape[0]} rows")
        display(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field} (first 5):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df)
    else:
        print("No numeric fields were found in this dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if primary_record_set_id and numeric_candidates:
    df = dataframes[primary_record_set_id]
    numeric_field = numeric_candidates[0]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    # If group candidate and at least two groups
    if group_candidates and df[group_candidates[0]].nunique() <= 8:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_candidates[0]], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_candidates[0]}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.